# CIFAR-10: multi-class (Part g) and class imbalance (Part h)

Applies the final Dogs vs. Cats model (Config C: ResNet-18, layer4 + fc fine-tuned) to CIFAR-10,
then simulates class imbalance and compares two fixes.

**How to run:** Kernel → Restart, then Run All.

- `FULL_RUN = False`: 300 train images per class, 100 test images per class, 2 epochs. For testing on a laptop.
- `FULL_RUN = True`: all 50,000 train / 10,000 test images, 10 epochs. **Only the trainer sets this.**

CIFAR-10 downloads itself into `data/` on the first run (about 170 MB, not committed).

| Config | Part | Training data | Fix |
|---|---|---|---|
| `cifar-bal` | g | balanced (5,000 per class) | none |
| `cifar-imb` | h | classes 0–2 cut to 20% (1,000 each) | none (shows the problem) |
| `cifar-imb-wloss` | h | same imbalanced data | weighted cross-entropy loss |
| `cifar-imb-sampler` | h | same imbalanced data | weighted random sampler |

Outputs per config (paths fixed by `common.py`): `outputs/results_<config>_seed42.csv` (one row per epoch),
`outputs/perclass_<config>_seed42.csv` (precision / recall / F1 per class), and
`checkpoints/resnet18_<config>_seed42.pth` (last epoch; shared Drive). Plus `outputs/cifar_perclass_recall.png`.

In [ ]:
FULL_RUN = True     # trainer only: set True for the final run
RUN_PART_H = True    # False = only the balanced Part g run

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Subset, WeightedRandomSampler
from torchvision import datasets, transforms

import common
from common import *

set_seed()
print("device:", DEVICE)

## Part g: CIFAR-10

**Dataset** ([Krizhevsky, 2009](https://www.cs.toronto.edu/~kriz/cifar.html)): 60,000 colour images of 32×32 pixels in
10 classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck). 50,000 training images
(5,000 per class) and 10,000 test images (1,000 per class), all labelled.

**Problem:** single-label classification into 10 classes. Each image shows one object; the model outputs 10 scores
and the highest one is the prediction.

**What changes compared with Dogs vs. Cats**

| | Dogs vs. Cats (Config C) | CIFAR-10 (this notebook) |
|---|---|---|
| Classes | 2 | 10 |
| Output layer | `Linear(512, 2)` | `Linear(512, 10)` |
| Image size | varied photos, resized and cropped to 224 | 32×32, upscaled to 224 |
| Train augmentation | rotation, flip, random resized crop, colour jitter | random crop (4-pixel padding) + flip at 32×32, then upscale |
| Loading | `ImageFolder` + custom test loader | `torchvision.datasets.CIFAR10` (test set has labels) |
| Evaluation | val set, best epoch kept | **test set, last epoch kept** (no model selection on test data) |
| Epochs | up to 15 | 10 |
| Metrics | accuracy | accuracy + per-class precision / recall / F1 |

Unchanged: ResNet-18 pretrained on ImageNet, only layer4 + fc trained, frozen BatchNorm outside layer4,
Adam lr = 1e-4, StepLR ×0.5 every 3 epochs, batch size 32, seed 42, cross-entropy loss.

In [ ]:
EPOCHS = 10 if FULL_RUN else 2
LR = 1e-4            # same as Config C (confirmed by the Dogs vs. Cats LR sweep)
STEP_SIZE = 3
GAMMA = 0.5

SUBSET_PER_CLASS = {"train": 300, "test": 100}   # used when FULL_RUN is False
MINORITY_CLASSES = [0, 1, 2]                     # airplane, automobile, bird
MINORITY_FRACTION = 0.2                          # keep 20% of their training images

DATA_ROOT = common.REPO_ROOT / "data"

### Transforms

CIFAR images are 32×32, but ResNet-18 was pretrained on 224×224 ImageNet images, so every image is
upscaled to 224×224. Augmentation is applied **before** upscaling, at the original resolution:
a random 32×32 crop from the image padded by 4 pixels (small shifts) and a horizontal flip.
Rotation and colour jitter from Dogs vs. Cats are left out: at 32×32 a rotation blurs the few pixels
there are, and the standard CIFAR augmentation is crop + flip. Normalisation uses the ImageNet mean and
standard deviation because the backbone was trained with them.
1. Why enlarge to 224: ResNet-18 learned its patterns on 224×224 photos and shrinks every image 32× inside the network (224 → 7). A 32×32 image would shrink to 1×1 pixel, losing all layout. Enlarging keeps objects at the size the network expects. It adds no detail; each pixel just gets bigger.
2. The two augmentations, so each epoch sees slightly different images:
- Shift: add a 4-pixel black border (40×40), then cut a random 32×32 piece, moving the object up to 4 pixels.
- Flip: mirror left/right. A car facing left is still a car.
They're done before enlarging because it's cheaper on tiny images, and it's the standard CIFAR recipe.
3. Why no rotation or colour changes: rotating a 32×32 image smears its few pixels and leaves black corners. Crop + flip is the well-tested standard. That's a design choice you can state in the report, not a rule.
4. Normalise is to subtract ImageNet's average colour and divide by its spread, for red, green and blue separately. The input numbers then look like what the pretrained network saw in training.

In [ ]:
normalize = transforms.Normalize(common.IMAGENET_MEAN, common.IMAGENET_STD)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize(common.IMG_SIZE),
    transforms.ToTensor(),
    normalize,
])
test_tf = transforms.Compose([
    transforms.Resize(common.IMG_SIZE),
    transforms.ToTensor(),
    normalize,
])

train_full = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=train_tf)
test_full = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=test_tf)
CLASS_NAMES = train_full.classes
train_targets = np.array(train_full.targets)
test_targets = np.array(test_full.targets)
print(f"train {len(train_full):,} | test {len(test_full):,} | classes {CLASS_NAMES}")

In [ ]:
def take_per_class(targets, counts, seed=SEED):
    """Reproducible random selection of counts[c] indices from each class c (sorted indices)."""
    rng = np.random.default_rng(seed)
    picked = []
    for c, n in counts.items():
        idx = np.flatnonzero(targets == c)
        picked.append(rng.permutation(idx)[:n])
    return np.sort(np.concatenate(picked)).tolist()


def class_counts(indices, targets):
    return np.bincount(targets[indices], minlength=len(CLASS_NAMES))


n_train = None if FULL_RUN else SUBSET_PER_CLASS["train"]
n_test = None if FULL_RUN else SUBSET_PER_CLASS["test"]
bal_counts = {c: n_train or int((train_targets == c).sum()) for c in range(len(CLASS_NAMES))}
test_counts = {c: n_test or int((test_targets == c).sum()) for c in range(len(CLASS_NAMES))}

train_bal_idx = take_per_class(train_targets, bal_counts)
test_idx = take_per_class(test_targets, test_counts)

train_bal = Subset(train_full, train_bal_idx)
test_loader = make_loader(Subset(test_full, test_idx), shuffle=False)

print(f"{'FULL' if FULL_RUN else 'SUBSET'} run | epochs {EPOCHS}")
pd.DataFrame({"class": CLASS_NAMES,
              "train (balanced)": class_counts(train_bal_idx, train_targets),
              "test": class_counts(test_idx, test_targets)})

In [ ]:
def make_optimizer(model, lr):
    """Adam on the trainable parameters only + the shared StepLR schedule (same as notebook 02)."""
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)
    return optimizer, scheduler


def release_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_config(config, train_ds, criterion, sampler=None):
    """Train Config C's setup on train_ds, evaluate on the test set, save per-class results."""
    set_seed()   # same starting weights for every config
    model = get_resnet18(num_classes=len(CLASS_NAMES), mode="finetune")
    optimizer, scheduler = make_optimizer(model, LR)
    train_loader = make_loader(train_ds, shuffle=True, sampler=sampler)
    history = fit(model, config, train_loader, test_loader, optimizer, criterion.to(DEVICE),
                  epochs=EPOCHS, scheduler=scheduler, keep_bn_train=("layer4",),
                  eval_split="test")   # test split: keeps the LAST epoch, no selection on test data
    y_true, y_pred = predict_labels(model, test_loader)
    per_class = save_per_class(y_true, y_pred, config, CLASS_NAMES)
    del model, optimizer, scheduler
    release_gpu()
    return history, per_class

### Part g run: balanced CIFAR-10

Reported number for Part g: test accuracy after the last epoch.

In [ ]:
hist_bal, per_bal = run_config("cifar-bal", train_bal, nn.CrossEntropyLoss())
print(f"\nPart g test accuracy (last epoch): {hist_bal.eval_acc.iloc[-1]:.4f}")
per_bal

## Part h: class imbalance

**Setup:** classes 0–2 (airplane, automobile, bird) keep only 20% of their training images (1,000 each on the full run);
classes 3–9 keep all 5,000. The test set stays balanced, so every class is judged on the same number of images.

**Why accuracy alone is not enough:** with 38,000 training images of which only 3,000 are minority classes,
a model can ignore the rare classes and still score well overall. So every config is compared on
**per-class recall** (the share of each class's test images found) and **macro recall / macro F1**
(the plain average over classes, so a rare class counts as much as a common one).

| Config | What it tests |
|---|---|
| `cifar-imb` | the problem: same training as Part g, imbalanced data |
| `cifar-imb-wloss` | fix 1: weighted cross-entropy (cost-sensitive learning) |
| `cifar-imb-sampler` | fix 2: weighted random sampler (re-sampling) |

All three use the same seed, model, epochs and schedule as `cifar-bal`; only the data or the fix changes.

In [ ]:
imb_counts = {c: (round(n * MINORITY_FRACTION) if c in MINORITY_CLASSES else n)
              for c, n in bal_counts.items()}
train_imb_idx = take_per_class(train_targets, imb_counts)
train_imb = Subset(train_full, train_imb_idx)
imb_class_n = class_counts(train_imb_idx, train_targets)

pd.DataFrame({"class": CLASS_NAMES, "balanced": class_counts(train_bal_idx, train_targets),
              "imbalanced": imb_class_n})

In [ ]:
if RUN_PART_H:
    hist_imb, per_imb = run_config("cifar-imb", train_imb, nn.CrossEntropyLoss())

### Fix 1: weighted cross-entropy loss

Each class gets a loss weight inversely proportional to how many training images it has:

$$w_c = \frac{N}{K \, n_c}$$

N = training images, K = 10 classes, n_c = images of class c. On the full run that is 0.76 for the
common classes and 3.8 for the rare ones, so a mistake on a rare class costs 5× more. The weights average
to 1 per training image, so the overall loss scale (and the learning rate) stays comparable.

In [ ]:
class_weights = torch.tensor(len(train_imb_idx) / (len(CLASS_NAMES) * imb_class_n), dtype=torch.float32)
print(pd.Series(class_weights.numpy().round(3), index=CLASS_NAMES).to_string())

if RUN_PART_H:
    hist_wloss, per_wloss = run_config("cifar-imb-wloss", train_imb, nn.CrossEntropyLoss(weight=class_weights))

### Fix 2: weighted random sampler

Instead of changing the loss, change what the model sees. Each training image is drawn with probability
proportional to 1 / (images in its class), with replacement, so every class makes up about 10% of each epoch.
The epoch length stays the same (one draw per imbalanced training image). Rare-class images are seen about
3.8× per epoch on average; the random crop and flip make the repeats look slightly different, which limits
memorising them.

In [ ]:
sample_weights = (1.0 / imb_class_n)[train_targets[train_imb_idx]]
sampler = WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double),
                                num_samples=len(train_imb_idx), replacement=True,
                                generator=torch.Generator().manual_seed(SEED))

if RUN_PART_H:
    hist_sampler, per_sampler = run_config("cifar-imb-sampler", train_imb, nn.CrossEntropyLoss(),
                                           sampler=sampler)

### Fix 3: focal loss

Focal loss ([Lin et al., 2017](https://arxiv.org/abs/1708.02002)) scales each image's cross-entropy loss by how
wrong the model still is on it:

$$FL = -(1 - p_t)^{\gamma} \log p_t$$

p_t = the probability the model gives to the correct class, γ (gamma) = 2, the value recommended in the paper.
An easy image the model already gets right (p_t = 0.9) keeps only 1% of its loss; a hard one (p_t = 0.3) keeps 49%.
Training effort shifts to the hard images, and on imbalanced data most hard images belong to the rare classes.

Difference from fix 1: weighted loss uses the class counts; focal loss ignores the counts and looks at each image's
difficulty instead, so it also helps with hard images in the common classes. Note: this run's `test loss` column is
the focal loss, so compare it with the other runs on accuracy and recall, not loss.

In [ ]:
import torch.nn.functional as F


class FocalLoss(nn.Module):
    """Focal loss (Lin et al., 2017): cross-entropy scaled by (1 - p_true) ** gamma, averaged over the batch."""

    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction="none")   # -log p_true, one value per image
        p_true = torch.exp(-ce)
        return ((1 - p_true) ** self.gamma * ce).mean()


FOCAL_GAMMA = 2.0

if RUN_PART_H:
    hist_focal, per_focal = run_config("cifar-imb-focal", train_imb, FocalLoss(FOCAL_GAMMA))

### Fix 4: logit adjustment (no retraining)

A model trained on imbalanced data learns a built-in bias: it adds a little extra score to common classes,
roughly in proportion to the logarithm of how common they were in training. Logit adjustment
([Menon et al., 2021](https://arxiv.org/abs/2007.07314)) removes that bias **after training**, at prediction time:

$$\hat{y} = \arg\max_c \left( z_c - \tau \log \pi_c \right)$$

z_c = the model's raw score (logit) for class c, π_c (pi) = the share of class c in the imbalanced training set,
τ (tau) = 1. On the full run, π is 1,000 / 38,000 for each rare class and 5,000 / 38,000 for each common class, so
the rare classes get about +1.6 extra score relative to the common ones before the highest score is picked.

It reuses the `cifar-imb` model saved in `checkpoints/` and only re-scores the test images: no extra training, so
it costs seconds. It shows that part of the imbalance damage is a **bias in the final decision**, not lost knowledge.

In [ ]:
TAU = 1.0

if RUN_PART_H:
    prior = torch.tensor(imb_class_n / imb_class_n.sum(), dtype=torch.float32, device=DEVICE)
    adjustment = TAU * torch.log(prior)    # subtracted from every score below
    print("extra score per class (relative to the most common class):")
    print(pd.Series((adjustment.max() - adjustment).cpu().numpy().round(2), index=CLASS_NAMES).to_string())

    model = load_checkpoint(get_resnet18(num_classes=len(CLASS_NAMES), mode="finetune"),
                            checkpoint_path("resnet18", "cifar-imb"))
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in test_loader:
            logits = model(x.to(DEVICE))
            y_pred.append((logits - adjustment).argmax(1).cpu().numpy())
            y_true.append(y.numpy())
    per_logitadj = save_per_class(np.concatenate(y_true), np.concatenate(y_pred), "cifar-imb-logitadj", CLASS_NAMES)
    del model
    release_gpu()

## Summary

Reads the saved per-class files, so it also works after a restart without retraining.
Overall accuracy is computed from per-class recall × test images per class.

In [ ]:
CONFIGS = {
    "cifar-bal": "none (balanced data, Part g)",
    "cifar-imb": "none (imbalanced data)",
    "cifar-imb-wloss": "fix 1: loss, class weights",
    "cifar-imb-sampler": "fix 2: data, re-sampling",
    "cifar-imb-focal": "fix 3: loss, focus on hard images",
    "cifar-imb-logitadj": "fix 4: prediction, score adjustment",
}
per = {c: pd.read_csv(OUTPUT_DIR / f"perclass_{c}_seed42.csv")
       for c in CONFIGS if (OUTPUT_DIR / f"perclass_{c}_seed42.csv").exists()}

rows = []
for config, df in per.items():
    cls = df[df["class"] != "MACRO_AVG"].reset_index(drop=True)
    rows.append({
        "config": config,
        "fix": CONFIGS[config],
        "accuracy": (cls.recall * cls.support).sum() / cls.support.sum(),
        "macro recall": cls.recall.mean(),
        "macro F1": cls.f1.mean(),
        "minority recall (0-2)": cls.recall[MINORITY_CLASSES].mean(),
        "majority recall (3-9)": cls.recall.drop(MINORITY_CLASSES).mean(),
    })
summary = pd.DataFrame(rows).round(4)
summary

In [ ]:
recall = pd.DataFrame({c: df[df["class"] != "MACRO_AVG"].set_index("class").recall for c, df in per.items()})
ax = recall.plot.bar(figsize=(13, 4.5), width=0.85)
ax.set_ylabel("test recall")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.set_title(f"CIFAR-10 per-class test recall ({'full' if FULL_RUN else 'subset'} run); classes 0-2 are the minority")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cifar_perclass_recall.png", dpi=150)
plt.show()